# Task E - YOLO11n fine-tuning (Config A vs Config B)

**Status: PREPARED, NOT AUTHORISED.** Dataset `yolo-rock-v1.1` is provisional and not frozen.
Every training cell refuses to run unless `AUTHORIZED = True` below **and** `ARTEMIS_TRAINING_AUTHORIZED=1` is set
in the environment, after explicit human approval. Plan: `backend/training/configs/task_e_experiments.yaml`.

In [ ]:
import os, sys, json
from pathlib import Path
BACKEND = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
BACKEND = BACKEND if (BACKEND / 'training').exists() else BACKEND.parent
sys.path.insert(0, str(BACKEND))
AUTHORIZED = False   # set True only after written human approval of the dataset freeze and Task E

## 1. Environment (metadata only; set `compute=True` at the start of an authorised session)

In [ ]:
from training.env_check import check
env = check(compute=False)
print(json.dumps({k: v for k, v in env.items() if k != 'torch_arch_list'}, indent=1))
assert env['verdict'] == 'PASS'

## 2. Resolve all arms (dry run - no GPU)

In [ ]:
from training import train as T
arms = ['A_steps', 'B_steps', 'A_epochs']
seeds = [0, 1, 2]
plan = [T.resolve(a, s) for a in arms for s in seeds]
for m in plan:
    u = m['ultralytics_args']
    print(m['run'], u['data'].split('/')[-1].split('\\')[-1], 'epochs', u['epochs'], 'seed', u['seed'], 'weights', m['weights_sha256'][:12])

## 3. Train (guarded). Resumes automatically from `weights/last.pt` if a run was interrupted.

In [ ]:
if not AUTHORIZED or os.environ.get('ARTEMIS_TRAINING_AUTHORIZED') != '1':
    raise SystemExit('Training not authorised - stopping here.')
for m in plan:
    T.train(m)          # writes experiments/task-e/runs/<run>/ and appends experiments/task-e/runs_registry.jsonl

## 4. Resource usage of a run

In [ ]:
import csv
import matplotlib.pyplot as plt
run = 'A_steps_s0'
p = BACKEND.parent / 'experiments/task-e/runs' / run / 'resources.csv'
if p.exists():
    rows = list(csv.DictReader(p.open()))
    t = [float(r['t_s']) / 60 for r in rows]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(t, [float(r['gpu_mem_used_mb'] or 0) for r in rows]); ax[0].set(title='GPU memory (MB)', xlabel='min')
    ax[1].plot(t, [float(r['gpu_util_pct'] or 0) for r in rows]); ax[1].set(title='GPU util (%)', xlabel='min')
else:
    print('no run yet')